In [3]:
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

files = {
    "COVID-19": "covid_positivity.json",
    "Influenza": "influenza_positivity.json",
    "RSV": "RSV_positivity.json",
}

# every month of every year, in order
timeline = []
for year in years:
    for month in months:
        timeline.append((year, month))

# one average for every month on the timeline
def monthly_row(filename):
    with open(filename) as f:
        records = json.load(f)

    totals = {}
    counts = {}
    for key in timeline:
        totals[key] = 0
        counts[key] = 0

    for record in records:
        key = (record["year"], record["month"])
        if key in totals:
            totals[key] = totals[key] + record["metric_value"]
            counts[key] = counts[key] + 1

    row = []
    for key in timeline:
        if counts[key] > 0:
            row.append(totals[key] / counts[key])
        else:
            row.append(float("nan"))
    return row

# work out each disease's row once
all_rows = {}
for disease in files:
    all_rows[disease] = monthly_row(files[disease])

# label each January with its year
tick_positions = []
tick_labels = []
for i in range(len(timeline)):
    if timeline[i][1] == 1:
        tick_positions.append(i)
        tick_labels.append(timeline[i][0])

def draw_heatmap(selected):
    if len(selected) == 0:
        print("Select at least one disease")
        return

    grid = []
    for disease in selected:
        grid.append(all_rows[disease])

    plt.figure(figsize=(14, 1.5 + 1.2 * len(selected)))
    image = plt.imshow(grid, cmap="YlOrRd", aspect="auto")

    plt.xticks(tick_positions, tick_labels)
    plt.yticks(range(len(selected)), selected)
    plt.xlabel("Year (each column is one month)")
    plt.title("PCR Positivity by Month (2020 - 2026)")
    plt.colorbar(image, label="Average PCR positivity (%)")
    plt.show()

selector = widgets.SelectMultiple(
    options=list(files),
    value=list(files),
    description="Diseases",
)

widgets.interact(draw_heatmap, selected=selector)

interactive(children=(SelectMultiple(description='Diseases', index=(0, 1, 2), options=('COVID-19', 'Influenza'…

<function __main__.draw_heatmap(selected)>

In [6]:
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
england_population = 57000000   # approximate, check the figure for England (ONS)

files = {
    "COVID-19": "covid_admissions.json",
    "Influenza": "influenza_admissions.json",
    "RSV": "RSV_cases.json",
}

# multiply each value by this to get a weekly rate per 100,000
scale = {
    "COVID-19": 7 / england_population * 100000,   # daily count -> weekly rate
    "Influenza": 1,
    "RSV": 1,
}

# every month of every year, in order
timeline = []
for year in years:
    for month in months:
        timeline.append((year, month))

# one average for every month on the timeline
def monthly_row(filename, factor):
    with open(filename) as f:
        records = json.load(f)

    totals = {}
    counts = {}
    for key in timeline:
        totals[key] = 0
        counts[key] = 0

    for record in records:
        key = (record["year"], record["month"])
        if key in totals:
            totals[key] = totals[key] + record["metric_value"] * factor
            counts[key] = counts[key] + 1

    row = []
    for key in timeline:
        if counts[key] > 0:
            row.append(totals[key] / counts[key])
        else:
            row.append(float("nan"))
    return row

# work out each disease's row once
all_rows = {}
for disease in files:
    all_rows[disease] = monthly_row(files[disease], scale[disease])

# label each January with its year
tick_positions = []
tick_labels = []
for i in range(len(timeline)):
    if timeline[i][1] == 1:
        tick_positions.append(i)
        tick_labels.append(timeline[i][0])

def draw_heatmap(selected):
    if len(selected) == 0:
        print("Select at least one disease")
        return

    grid = []
    for disease in selected:
        grid.append(all_rows[disease])

    plt.figure(figsize=(14, 1.5 + 1.2 * len(selected)))
    image = plt.imshow(grid, cmap="YlOrRd", aspect="auto")

    plt.xticks(tick_positions, tick_labels)
    plt.yticks(range(len(selected)), selected)
    plt.xlabel("Year  (month per column)")
    plt.title("Hospital Admissions by Month (2020 - 2026)")
    plt.colorbar(image, label="Average weekly admission rate per 100,000")
    plt.show()

selector = widgets.SelectMultiple(
    options=list(files),
    value=list(files),
    description="Diseases",
)

widgets.interact(draw_heatmap, selected=selector)

interactive(children=(SelectMultiple(description='Diseases', index=(0, 1, 2), options=('COVID-19', 'Influenza'…

<function __main__.draw_heatmap(selected)>

Heat map shows hospital admissions each month from 2020 - 2026 for COVID-19, Influenza and RSV. 

COVID-19 hospital admissions greater in 2020.
Heatmap tends to be warmer towards winter times (seasonal).  